<a href="https://colab.research.google.com/github/zach-yan/ORFE-Network-Project/blob/main/ORF_387_Semantic_Scholar_Network_Creation_and_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Semantic Scholar API key

In [ ]:
# Add an API key if you have one
API_KEY = "6M9qBa9XS79hpcxQloIqo2iyNYVHTaIW5T4vg3Sn" # Insert your Semantic Scholar API key here
HEADERS = {"x-api-key": API_KEY} if API_KEY else {}

Initial network creation via a BFS algorithm originating from ORFE faculty Semantic Scholar IDs

In [ ]:
import requests
import time
import networkx as nx
import pandas as pd
from google.colab import files, userdata
from tqdm.auto import tqdm

# 1. Fetch the API key from Colab Secrets
try:
    API_KEY = userdata.get('SEMANTIC_SCHOLAR_API_KEY')
    HEADERS = {"x-api-key": API_KEY}
    print("API Key loaded successfully!")
except userdata.SecretNotFoundError:
    print("Warning: API Key not found in Colab Secrets. Running unauthenticated.")
    HEADERS = {}

# 2. Initialize graph
G = nx.Graph()

# Layer 0 IDs
princeton_seed_ids = [
    '2310028016','79676449', '2294567016', '8386952', '1792465',
    '2344786184', '25681050', '9051324', '1684820', '12890978',
    '2242596739', '1697174', '1825721854', '2092062', '38967552',
    '102254552', '144958605', '2266124380', '7402018', '33933730',
    '2066731293', '2066733747', '1697413', '17434392',
    '30030844', '2248191566', '2258225151'
]

# Constraints
max_depth = 2
recent_year_cutoff = 2018
max_authors_per_paper = 15

# 3. Upgraded fetch function with error handling and progress bar
def fetch_authors_batch(author_ids, fields, layer_name):
    url = f"https://api.semanticscholar.org/graph/v1/author/batch?fields={fields}"
    results = []
    chunk_size = 50

    for i in tqdm(range(0, len(author_ids), chunk_size), desc=f"Fetching {layer_name}"):
        raw_chunk = author_ids[i:i+chunk_size]
        chunk = [aid for aid in raw_chunk if aid and isinstance(aid, str) and aid.strip()]
        if not chunk: continue

        payload = {"ids": chunk}

        success = False
        retries = 0
        max_retries = 3

        while not success and retries < max_retries:
            try:
                response = requests.post(url, json=payload, headers=HEADERS, timeout=45)

                if response.status_code == 200:
                    results.extend([r for r in response.json() if r is not None])
                    success = True
                elif response.status_code == 429:
                    tqdm.write(f"  [!] Rate limited. Sleeping for 15s...")
                    time.sleep(15)
                elif response.status_code == 400:
                    tqdm.write(f"  [!] Error 400 (Bad Request). Skipping this chunk.")
                    success = True
                elif response.status_code in [500, 502, 503, 504]:
                    retries += 1
                    tqdm.write(f"  [!] Server Timeout/Error {response.status_code}. Retry {retries}/{max_retries} in 10s...")
                    time.sleep(10)
                else:
                    tqdm.write(f"  [!] Unexpected Error {response.status_code}. Skipping chunk.")
                    success = True

            except requests.exceptions.RequestException as e:
                retries += 1
                tqdm.write(f"  [!] Network error: {e}. Retry {retries}/{max_retries} in 5s...")
                time.sleep(5)

        # Hard stop for 1.1 seconds between every single batch request
        time.sleep(1.1)

    return results

print(f"Starting Batch BFS to Depth {max_depth} with Year Cutoff >= {recent_year_cutoff}...\n")
print("-" * 50)

visited = set(princeton_seed_ids)
seen_collaborations = set()

current_layer_ids = princeton_seed_ids
depth = 0

# 4. Main Layer-by-Layer Processing
while current_layer_ids and depth <= max_depth:
    print(f"Processing Layer {depth} ({len(current_layer_ids)} nodes)...")

    if depth == max_depth:
        fields = "authorId,name,affiliations"
    else:
        fields = "authorId,name,affiliations,papers.year,papers.authors,papers.paperId"

    author_data_list = fetch_authors_batch(current_layer_ids, fields, layer_name=f"Layer {depth}")
    next_layer_ids = set()

    for author_data in author_data_list:
        current_id = author_data.get('authorId')
        if not current_id: continue

        current_name = author_data.get('name', 'Unknown')
        affiliations = author_data.get('affiliations', [])
        current_affiliation = affiliations[0] if affiliations else "Unknown"

        # Kept Bertsekas affiliation as MIT here
        if "Bertsekas" in current_name:
            current_affiliation = "MIT"

        # Add node
        G.add_node(current_id, name=current_name, layer=depth, affiliation=current_affiliation)

        if depth < max_depth:
            papers = author_data.get('papers', [])

            for paper in papers:
                paper_id = paper.get('paperId')
                year = paper.get('year')
                authors = paper.get('authors', [])

                if not paper_id or year is None or year < recent_year_cutoff or len(authors) > max_authors_per_paper:
                    continue

                for coauthor in authors:
                    coauthor_id = coauthor.get('authorId')

                    if coauthor_id and coauthor_id != current_id:
                        collab_key = (frozenset([current_id, coauthor_id]), paper_id)

                        if collab_key not in seen_collaborations:
                            if G.has_edge(current_id, coauthor_id):
                                G[current_id][coauthor_id]['weight'] += 1
                            else:
                                G.add_edge(current_id, coauthor_id, weight=1)
                            seen_collaborations.add(collab_key)

                        if coauthor_id not in visited:
                            visited.add(coauthor_id)
                            next_layer_ids.add(coauthor_id)

    current_layer_ids = list(next_layer_ids)
    depth += 1

print("-" * 50)
print(f"Final Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

# 5. Export
def export_weighted_network(G):
    nodes_df = pd.DataFrame([{
        "Id": n,
        "Label": d.get('name'),
        "Layer": d.get('layer'),
        "Affiliation": d.get('affiliation')
    } for n, d in G.nodes(data=True)])

    edges_df = pd.DataFrame([{
        "Source": u,
        "Target": v,
        "Weight": d.get('weight', 1),
        "Type": "Undirected"
    } for u, v, d in G.edges(data=True)])

    nodes_df.to_csv('orfe_weighted_nodes2.csv', index=False)
    edges_df.to_csv('orfe_weighted_edges2.csv', index=False)

    files.download('orfe_weighted_nodes2.csv')
    files.download('orfe_weighted_edges2.csv')
    print("Files downloaded successfully.")

export_weighted_network(G)

Clean for exact duplicates and output the node mergings

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Load your generated datasets
nodes_df = pd.read_csv('orfe_weighted_nodes.csv')
edges_df = pd.read_csv('orfe_weighted_edges.csv')

# 1. Identify Duplicates
# We group by Label (Name) and collect all unique IDs associated with that name
name_groups = nodes_df.groupby('Label')['Id'].unique()
duplicates = name_groups[name_groups.map(len) > 1]

print("--- DUPLICATE MERGING REPORT ---")
if duplicates.empty:
    print("No duplicate names found. Everything looks clean!")
else:
    for name, ids in duplicates.items():
        print(f"Combining {len(ids)} profiles for: {name}")
        print(f"  -> IDs being merged: {', '.join(map(str, ids))}")
print("-" * 32 + "\n")

# 2. Map duplicate IDs to a single Primary ID
# Sorting by Layer ensures the primary ID is the one closest to the seed nodes (Layer 0)
nodes_df = nodes_df.sort_values('Layer')
name_to_primary_id = nodes_df.groupby('Label')['Id'].first().to_dict()

# Create a dictionary mapping every old ID to its new Primary ID
id_to_primary_map = nodes_df.set_index('Id')['Label'].map(name_to_primary_id).to_dict()

# 3. Rewire the edges
# Replace the Source and Target IDs with the "Primary" version of those names
edges_df['Source'] = edges_df['Source'].map(id_to_primary_map)
edges_df['Target'] = edges_df['Target'].map(id_to_primary_map)

# 4. Clean up the edges
# Standardize direction so (A->B) and (B->A) are treated identically
edges_df[['Source', 'Target']] = np.sort(edges_df[['Source', 'Target']], axis=1)

# Remove self-loops (authors co-authoring with their own duplicate profiles)
edges_df = edges_df[edges_df['Source'] != edges_df['Target']]

# Group duplicate edges and sum their weights
edges_df = edges_df.groupby(['Source', 'Target', 'Type'], as_index=False)['Weight'].sum()

# 5. Clean up the nodes
# Keep only one row per name, prioritizing the lowest Layer and first Affiliation found
nodes_df = nodes_df.groupby('Label', as_index=False).agg({
    'Id': 'first',
    'Layer': 'min',
    'Affiliation': 'first'
})

# Reorder columns to match the original format
nodes_df = nodes_df[['Id', 'Label', 'Layer', 'Affiliation']]

# 6. Export the clean graph
nodes_df.to_csv('orfe_nodes_cleaned.csv', index=False)
edges_df.to_csv('orfe_edges_cleaned.csv', index=False)

files.download('orfe_nodes_cleaned.csv')
files.download('orfe_edges_cleaned.csv')

print(f"Final Count: {len(nodes_df)} unique nodes, {len(edges_df)} consolidated edges.")

We then clean the nodes further to consolidate nodes into their longest possible name (e.g., *J. Doe* to *John Doe*) and output the result.

In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files

# Load your generated datasets
nodes_df = pd.read_csv('orfe_nodes_cleaned.csv')
edges_df = pd.read_csv('orfe_edges_cleaned.csv')

def normalize(n): return re.sub(r'\.', '', n).lower().split()

def get_name_parts(name):
    """Cleans a name and splits it into (First_Initial, Last_Name, Full_Name)."""
    clean = re.sub(r'\.', '', str(name)).lower().strip()
    parts = clean.split()
    if len(parts) < 2:
        return (clean, clean, name)
    return (parts[0][0], parts[-1], name)

# 1. Create a lookup table of potential 'Full Name' candidates
# We group by (First_Initial, Last_Name)
author_info = nodes_df['Label'].unique()
candidates = {}
for name in author_info:
    initial, last, full = get_name_parts(name)
    key = (initial, last)
    if key not in candidates:
        candidates[key] = []
    candidates[key].append(name)

# 2. Build the Fuzzy Mapping
fuzzy_map = {} # Maps 'Initial Version' -> 'Fullest Version'
print("--- FUZZY MERGING REPORT ---")

for key, names in candidates.items():
    if len(names) > 1:
        # Sort names by length to find the 'Fullest' one
        # e.g., ['J. Doe', 'John Doe'] -> 'John Doe' is the winner
        fullest_name = sorted(names, key=len, reverse=True)[0]

        # We only merge if the 'fullest' name actually contains more info
        # than the other entries (i.e., it's not just a list of identical names)
        for name in names:
            if name != fullest_name:
                # Simple safety check: don't merge if they are both full names
                # but different (e.g., 'James Doe' and 'John Doe')
                p1 = normalize(name)
                p2 = normalize(fullest_name)
                if len(p1[0]) == 1 or p1[0] == p2[0]: # If one is an initial or they match
                    fuzzy_map[name] = fullest_name
                    print(f"Expanding: '{name}' → '{fullest_name}'")


print("-" * 32 + "\n")

# 3. Apply Fuzzy Mapping to Nodes
nodes_df['Label'] = nodes_df['Label'].replace(fuzzy_map)

# 4. Collapse IDs (Standard exact-match logic from before)
nodes_df = nodes_df.sort_values('Layer')
name_to_primary_id = nodes_df.groupby('Label')['Id'].first().to_dict()
id_to_primary_map = nodes_df.set_index('Id')['Label'].map(name_to_primary_id).to_dict()

# 5. Rewire and aggregate edges
edges_df['Source'] = edges_df['Source'].map(id_to_primary_map)
edges_df['Target'] = edges_df['Target'].map(id_to_primary_map)
edges_df[['Source', 'Target']] = np.sort(edges_df[['Source', 'Target']], axis=1)
edges_df = edges_df[edges_df['Source'] != edges_df['Target']]
edges_df = edges_df.groupby(['Source', 'Target', 'Type'], as_index=False)['Weight'].sum()

# 6. Final Node Consolidation
nodes_df = nodes_df.groupby('Label', as_index=False).agg({
    'Id': 'first',
    'Layer': 'min',
    'Affiliation': 'first'
})
nodes_df = nodes_df[['Id', 'Label', 'Layer', 'Affiliation']]

# 7. Export
nodes_df.to_csv('orfe_nodes_fuzzy_cleaned.csv', index=False)
edges_df.to_csv('orfe_edges_fuzzy_cleaned.csv', index=False)
files.download('orfe_nodes_fuzzy_cleaned.csv')
files.download('orfe_edges_fuzzy_cleaned.csv')

print(f"Final Count: {len(nodes_df)} unique nodes, {len(edges_df)} consolidated edges.")